# 🧠 Agent Memory Search: Personalized Banking Assistant

Welcome to our **Agent Memory Search** tutorial! This notebook demonstrates how to build an AI banking assistant that **remembers customer preferences** across conversations using the **Memory Search Tool**.

## What You'll Build

A **Personalized Banking Advisor** that:
- 🧠 **Remembers** customer preferences (risk tolerance, account types, communication preferences)
- 💬 **Recalls** past interactions in new conversations
- 🏦 **Provides personalized** financial guidance based on stored memories
- 📊 **Maintains context** across multiple banking sessions

## Why Memory Matters in Banking

| Without Memory | With Memory |
|----------------|-------------|
| Customer repeats preferences every call | Agent recalls "You prefer low-risk investments" |
| Generic product recommendations | Personalized suggestions based on history |
| No context from previous sessions | Seamless continuation of past discussions |
| Impersonal experience | "Welcome back! Last time we discussed..." |

## Architecture Overview

```
┌──────────────────────────────────────────────────────────────────────┐
│                    MEMORY-ENABLED BANKING AGENT                       │
├──────────────────────────────────────────────────────────────────────┤
│                                                                       │
│   Conversation 1                    Conversation 2                    │
│   ┌─────────────┐                  ┌─────────────┐                   │
│   │ "I prefer   │                  │ "What should│                   │
│   │  low-risk   │                  │  I invest   │                   │
│   │  investments"│                 │  in?"       │                   │
│   └──────┬──────┘                  └──────┬──────┘                   │
│          │                                │                          │
│          ▼                                ▼                          │
│   ┌─────────────┐                  ┌─────────────┐                   │
│   │   Memory    │◄────────────────►│   Memory    │                   │
│   │   Store     │   Automatic      │   Search    │                   │
│   │  (Extract)  │   Retrieval      │  (Recall)   │                   │
│   └──────┬──────┘                  └──────┬──────┘                   │
│          │                                │                          │
│          ▼                                ▼                          │
│   ┌──────────────────────────────────────────────┐                   │
│   │           Azure AI Memory Store               │                   │
│   │  • User Profile (preferences, risk tolerance) │                   │
│   │  • Chat Summary (conversation highlights)     │                   │
│   │  • Semantic Search (find relevant memories)   │                   │
│   └──────────────────────────────────────────────┘                   │
│                                                                       │
└──────────────────────────────────────────────────────────────────────┘
```

## Prerequisites

- Microsoft Foundry project with appropriate permissions
- **Chat model deployment** (e.g., `gpt-4o`)
- **Embedding model deployment** (e.g., `text-embedding-3-large`)
- A `.env` file in the project root containing (uses existing variables):
  ```bash
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
  EMBEDDING_MODEL_DEPLOYMENT_NAME=text-embedding-3-large
  TENANT_ID=<your-azure-tenant-id>
  ```

## Key Concepts

| Concept | Description |
|---------|-------------|
| **Memory Store** | Persistent storage for extracted memories from conversations |
| **Memory Search Tool** | Agent tool that retrieves relevant memories during conversations |
| **Scope** | User identifier to isolate memories per customer |
| **User Profile** | Automatically extracted preferences and traits |
| **Chat Summary** | Condensed highlights from past conversations |

### ⚠️ Important Financial Disclaimer
> **The financial information provided by this notebook is for general educational and demonstration purposes only.** Always consult with qualified financial advisors before making financial decisions.

📚 **Reference**: [Azure AI Agent Memory Documentation](https://learn.microsoft.com/azure/ai-services/agents/concepts/memory)

---

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI:

```bash
az login --use-device-code
```

This provides device code authentication, useful for:
- Remote development environments
- Systems without a default browser
- Corporate environments with strict security policies

---

## 📦 Step 1: Install Required Packages

The required packages should already be installed via the `requirements.txt` file at the project root:

```bash
pip install -r requirements.txt
```

If packages are not installed, uncomment and run the cell below.

In [ ]:
# Packages should be installed via requirements.txt at project root
# If not installed, uncomment the line below and run this cell:

# %pip install azure-ai-projects>=2.0.0b1 azure-identity openai python-dotenv --quiet

print("✅ Packages ready (installed via requirements.txt)")

---

## 🔧 Step 2: Load Configuration and Initialize Clients

Load environment variables and establish connections to Microsoft Foundry.

In [ ]:
import os
import time
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MemorySearchPreviewTool,
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
    PromptAgentDefinition,
)
from azure.identity import AzureCliCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

tenant_id = os.getenv("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
embedding_deployment = os.getenv("EMBEDDING_MODEL_DEPLOYMENT_NAME")
required_settings = {
    "TENANT_ID": tenant_id,
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_deployment,
    "EMBEDDING_MODEL_DEPLOYMENT_NAME": embedding_deployment,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

memory_update_delay = int(os.getenv("MEMORY_UPDATE_DELAY_SECONDS", "5"))
if memory_update_delay < 1:
    raise RuntimeError("MEMORY_UPDATE_DELAY_SECONDS must be at least 1.")
run_suffix = uuid4().hex[:8]
memory_chat_model = model_deployment
memory_embedding_model = embedding_deployment

print(f"Memory notebook configuration loaded for run {run_suffix}")

In [ ]:
# Initialize AIProjectClient with Azure CLI authentication
print("🔐 Initializing clients with AzureCliCredential...")
print("   Make sure you've run 'az login' in your terminal first!")
print("")

credential = AzureCliCredential(tenant_id=tenant_id)

# Create the project client (allow_preview needed for memory stores)
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    allow_preview=True
)

# Get the OpenAI client for conversations
openai_client = project_client.get_openai_client()

print("✅ AIProjectClient initialized (preview features enabled)")
print("✅ OpenAI client ready for conversations")

---

## 🗄️ Step 3: Create a Memory Store

The **Memory Store** is where customer memories are persisted. It uses:
- A **chat model** to extract meaningful memories from conversations
- An **embedding model** to enable semantic search across memories

### Memory Store Options:
| Option | Description |
|--------|-------------|
| `user_profile_enabled` | Extract and maintain user preferences/traits |
| `chat_summary_enabled` | Create summaries of past conversations |

In [ ]:
memory_store_name = f"banking-customer-memory-{run_suffix}"
memory_definition = MemoryStoreDefaultDefinition(
    chat_model=memory_chat_model,
    embedding_model=memory_embedding_model,
    options=MemoryStoreDefaultOptions(
        user_profile_enabled=True,
        chat_summary_enabled=True,
    ),
)
memory_store = project_client.beta.memory_stores.create(
    name=memory_store_name,
    description="Per-run memory store for the personalized banking assistant",
    definition=memory_definition,
)

print(f"Created memory store {memory_store.name} ({memory_store.id})")

---

## 🔧 Step 4: Configure the Memory Search Tool

The **Memory Search Tool** allows the agent to:
1. **Search** for relevant memories during conversations
2. **Update** memories after periods of inactivity

### Key Parameters:
| Parameter | Description |
|-----------|-------------|
| `memory_store_name` | The memory store to search |
| `scope` | User identifier (e.g., customer ID) to isolate memories |
| `update_delay` | Seconds of inactivity before extracting new memories |

In [ ]:
customer_id = f"customer-john-doe-{run_suffix}"
memory_tool = MemorySearchPreviewTool(
    memory_store_name=memory_store.name,
    scope=customer_id,
    update_delay=memory_update_delay,
)

print(f"Memory tool configured for scope {customer_id}")
print(f"Memory update delay: {memory_update_delay} seconds")

---

## 🤖 Step 5: Create the Memory-Enabled Banking Agent

Create a personalized banking assistant that uses memory to provide contextual, personalized financial guidance.

In [ ]:
# Agent name for the banking assistant
agent_name = "personalized-banking-advisor"

# Detailed instructions for the banking agent
agent_instructions = """
You are a Personalized Banking Advisor with memory capabilities. You remember customer preferences and past interactions to provide tailored financial guidance.

## YOUR CAPABILITIES:

🧠 **Memory-Enabled Personalization**
- You remember customer preferences shared in previous conversations
- You recall risk tolerance, investment preferences, and financial goals
- You reference past discussions to provide continuity

🏦 **Banking Services**
- Savings and checking account guidance
- Certificate of Deposit (CD) recommendations
- Money market account information
- General loan and mortgage information

📊 **Investment Guidance**
- Risk assessment discussions
- Portfolio diversification concepts
- Retirement planning basics (401k, IRA)
- Investment product education

## HOW TO RESPOND:

1. **Use Memory**: Reference what you remember about the customer:
   - "Based on your preference for low-risk investments..."
   - "Since you mentioned you're saving for retirement..."
   - "Given your conservative risk tolerance..."

2. **Ask About Preferences**: If you don't have relevant memories:
   - "What's your risk tolerance for investments?"
   - "What are your primary financial goals?"
   - "Do you prefer aggressive or conservative strategies?"

3. **Personalize Recommendations**: Tailor advice to stored preferences:
   - Match products to risk tolerance
   - Align suggestions with stated goals
   - Reference past discussions when relevant

## IMPORTANT DISCLAIMERS:

⚠️ Always include appropriate disclaimers:
- You are not a licensed financial advisor
- Recommendations are for educational purposes only
- Customers should consult qualified professionals for personalized advice
- Past performance does not guarantee future results

## EXAMPLE INTERACTIONS:

**First Conversation:**
Customer: "I'm looking to invest but I'm risk-averse"
You: Store this preference and suggest conservative options like bonds, CDs, high-yield savings

**Later Conversation:**
Customer: "What should I invest in?"
You: "Based on your conservative risk profile from our previous discussion, you might consider..."
"""

print(f"📝 Creating agent: {agent_name}")
print(f"   Instructions: {len(agent_instructions)} characters")

In [ ]:
# Create the agent with Memory Search Tool
print("🤖 Creating Memory-Enabled Banking Agent...")

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=agent_instructions,
        tools=[memory_tool]  # Attach the memory search tool
    )
)

print("\n" + "=" * 60)
print("✅ Agent Created with Memory Capabilities")
print("=" * 60)
print(f"🤖 Name: {agent.name}")
print(f"🆔 ID: {agent.id}")
print(f"📌 Version: {agent.version}")
print(f"🧠 Model: {model_deployment}")
print(f"🔧 Tools: Memory Search Tool")
print(f"\n🎯 Agent Capabilities:")
print(f"   • Remember customer preferences across conversations")
print(f"   • Recall past financial discussions")
print(f"   • Provide personalized banking guidance")

---

## 💬 Step 6: Establish Customer Preferences (Conversation 1)

In this first conversation, the customer will share their financial preferences. The agent will:
1. Respond helpfully
2. Store these preferences in memory for future use

In [ ]:
def chat_with_agent(conversation_id, message, agent_ref):
    """Send one message and require a text response."""
    response = openai_client.responses.create(
        input=message,
        conversation=conversation_id,
        extra_body={
            "agent_reference": {
                "type": "agent_reference",
                "name": agent_ref.name,
                "version": agent_ref.version,
            }
        },
    )
    if not response.output_text:
        raise RuntimeError("The memory-enabled agent returned no text output.")
    print(f"\nCustomer: {message}")
    print(f"Advisor: {response.output_text}")
    return response.output_text


print("Chat helper defined")

In [ ]:
# Create first conversation - establishing preferences
print("📞 Starting Conversation 1: Establishing Customer Preferences")
print("=" * 70)

conversation1 = openai_client.conversations.create()
print(f"✅ Conversation created (ID: {conversation1.id})")

# Customer shares their preferences
preference_messages = [
    "Hi, I'm a new customer. I'm 45 years old and I prefer conservative, low-risk investments. I'm saving for retirement in about 20 years.",
    "I also want to set up an emergency fund. I prefer to keep about 6 months of expenses in liquid savings.",
    "My annual income is around $120,000 and I can invest about $1,500 per month."
]

for msg in preference_messages:
    chat_with_agent(conversation1.id, msg, agent)
    print("\n" + "-" * 70)

print("\n" + "=" * 70)
print("✅ Conversation 1 Complete - Preferences Shared")
print("=" * 70)

---

## ⏳ Step 7: Wait for Memory Extraction

After a period of inactivity (defined by `update_delay`), the agent automatically extracts memories from the conversation and stores them.

**In this demo**: We wait a short time for memories to be extracted.  
**In production**: The `update_delay` would be longer (e.g., 5 minutes) to avoid extracting during active typing.

In [ ]:
memory_settle_seconds = memory_update_delay + 5
print(f"Waiting {memory_settle_seconds} seconds for memory extraction...")
time.sleep(memory_settle_seconds)
print("Memory extraction window completed")

---

## 🔄 Step 8: Test Memory Recall (Conversation 2)

Now we'll start a **brand new conversation**. The agent should recall the customer's preferences from the memory store and provide personalized responses.

In [ ]:
conversation2 = openai_client.conversations.create()
recall_messages = [
    "Hi, I spoke with someone earlier. Can you recommend some investment options for me?",
    "What about my emergency fund? How should I set that up?",
    "Given my situation, should I max out my 401k contributions?",
]
recall_responses = [
    chat_with_agent(conversation2.id, message, agent)
    for message in recall_messages
]

combined_recall = " ".join(recall_responses).lower()
original_memory_signals = [
    ("conservative", "low-risk"),
    ("retirement", "20 year"),
    ("emergency fund", "six month", "6 month"),
    ("120,000", "1,500"),
]
matched_signals = [
    signals
    for signals in original_memory_signals
    if any(signal in combined_recall for signal in signals)
]
if len(matched_signals) < 2:
    raise RuntimeError(
        "Memory recall was not demonstrated: fewer than two stored preference "
        "signals appeared in the new conversation."
    )

print(f"Verified {len(matched_signals)} original memory signal groups")

---

## 🔄 Step 9: Update Preferences (Conversation 3)

Customers' preferences can change over time. Let's test how the agent handles updated preferences.

In [ ]:
conversation3 = openai_client.conversations.create()
update_messages = [
    "I've been thinking, and I'd like to be a bit more aggressive with my investments. I can handle moderate risk now.",
    "I also got a raise! My income is now $140,000 and I can invest $2,000 per month.",
]
for message in update_messages:
    chat_with_agent(conversation3.id, message, agent)

print(f"Waiting {memory_settle_seconds} seconds for updated memory extraction...")
time.sleep(memory_settle_seconds)
print("Updated memory extraction window completed")

---

## 🧪 Step 10: Verify Updated Memory (Conversation 4)

Let's verify that the agent remembers the **updated** preferences.

In [ ]:
conversation4 = openai_client.conversations.create()
verify_message = (
    "Based on everything you know about me, what's your top investment "
    "recommendation right now? Include the profile details you used."
)
updated_recall = chat_with_agent(conversation4.id, verify_message, agent).lower()
updated_memory_signals = [
    ("moderate",),
    ("140,000", "140k"),
    ("2,000", "2000"),
]
missing_updated_signals = [
    signals
    for signals in updated_memory_signals
    if not any(signal in updated_recall for signal in signals)
]
if missing_updated_signals:
    raise RuntimeError(
        "Updated memory was not fully demonstrated. Missing signal groups: "
        f"{missing_updated_signals}"
    )

print("Verified updated risk, income, and monthly investment memories")

---

## 🧹 Step 11: Cleanup Resources

Delete all created resources to clean up.

In [ ]:
cleanup_errors = []

for conversation in [conversation1, conversation2, conversation3, conversation4]:
    try:
        openai_client.conversations.delete(conversation.id)
        print(f"Deleted conversation {conversation.id}")
    except Exception as error:
        cleanup_errors.append(f"conversation {conversation.id}: {error}")

try:
    project_client.agents.delete_version(
        agent_name=agent.name,
        agent_version=agent.version,
    )
    print(f"Deleted {agent.name} version {agent.version}")
except Exception as error:
    cleanup_errors.append(f"agent {agent.name} version {agent.version}: {error}")

try:
    project_client.beta.memory_stores.delete(memory_store.name)
    print(f"Deleted memory store {memory_store.name}")
except Exception as error:
    cleanup_errors.append(f"memory store {memory_store.name}: {error}")

openai_client.close()
project_client.close()
credential.close()

if cleanup_errors:
    raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))

---

## 📚 Summary

In this notebook, you learned how to build a **memory-enabled banking assistant** that provides personalized financial guidance by remembering customer preferences.

### What You Built

| Component | Description |
|-----------|-------------|
| **Memory Store** | Persistent storage for customer preferences and conversation summaries |
| **Memory Search Tool** | Agent tool that retrieves and updates memories |
| **Banking Agent** | AI assistant that uses memories for personalization |

### Key Concepts Demonstrated

1. **Memory Extraction** - Automatically extract preferences from conversations
2. **Memory Recall** - Retrieve relevant memories in new conversations
3. **Memory Updates** - Handle changing customer preferences over time
4. **Scoped Memories** - Isolate memories per customer for privacy

### Industry Use Cases for Agent Memory

| Use Case | Memory Benefit |
|----------|----------------|
| **Wealth Management** | Remember risk tolerance, investment preferences, financial goals |
| **Customer Service** | Recall past issues, preferred communication channels, account history |
| **Loan Processing** | Remember document submissions, employment history, property details |
| **Insurance** | Recall policy preferences, claims history, coverage requirements |
| **Fraud Detection** | Remember customer behavior patterns, typical transaction types |

### Production Considerations

| Consideration | Recommendation |
|---------------|----------------|
| **Update Delay** | Set to 300+ seconds (5 min) to avoid extraction during active conversations |
| **Scope Strategy** | Use customer IDs or `{{$userId}}` for automatic user isolation |
| **Memory Retention** | Consider data retention policies and GDPR compliance |
| **Sensitive Data** | Avoid storing PII or sensitive financial data in memories |

---

**🎉 Congratulations!** You've successfully built a personalized banking assistant with memory capabilities!

*Note: This is for educational purposes. Production deployments require proper error handling, security, and compliance features.*